In [ ]:
from pathlib import Path
import numpy as np
import polars as pl
from functools import partial
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit

from piepy.psychophysics.wheel.detection.wheelDetectionExperimentHub import WheelDetectionExperimentHub
from piepy.core.data_functions import make_subsets

from piepy.psychophysics.wheel.detection.wheelDetectionGroupedAggregator import WheelDetectionGroupedAggregator
from piepy.plotters.plotting_utils import set_style

In [ ]:
DATA_PATH = Path.cwd().parents[0] / "260410_Ncomm_inhibition_data.parquet"

CONTRAST_IDX = {0.0:0,
                0.125:1,
                0.5:2}

MANIP_LABEL = {0:"Control",
               1:"Laser ON",}

MANIP_MARKER = {0:"o",
                1:"x",}

STYPE_COLOR = {"0.04cpd_8.0Hz":"#FF7F0F",
               "0.16cpd_0.5Hz":"#0099C2"}

In [ ]:
def nan_generator(shape:tuple):
    """_summary_

    Args:
        shape (tuple): _description_

    Returns:
        _type_: _description_
    """
    x = np.zeros(shape)
    x[:] = np.nan
    return x

def sigmoid(x_arr, x0, L, k, p_false=0):
    """
    x0 = x value of the sigmoid midpoint
    L = max value of sigmoid
    k = logistic growth rate aka steepness
    b = y-offset(if needed)
    """
    return L / (1.0 + np.exp(-k * (x_arr - x0))) + p_false

def naka_rushton(c_arr,c50,n,p_max,p_false=0):
    """
    c_arr : contrast array
    n : slope/shape parameter
    c50 : contrast giving halfway point between p_false and p_max
    p_false : probability of saying “yes” when no stimulus is present
    p_max : hit rate at high contrast
    """
    return p_false + (p_max-p_false)/(1+(c50/c_arr)**n)

def fit_naka_rushton(lin_ax, responses, p_false, bounds=None):
    
    fit_func = naka_rushton
    
    if bounds is None:
        bounds = ([0.5, 1, 0], [1.5, 5, 1]) # [c50,n,p_max]
    
    fit_func_partial = partial(fit_func,p_false=p_false)
    popt,pcov = curve_fit(fit_func_partial, 
                            xdata=lin_ax, 
                            ydata=responses,
                            p0=[1,3,0], 
                            bounds=bounds, 
                            maxfev=100)
    interp_c50 = np.interp(popt[0],lin_ax,[0,12.5,50])
    fitted_p_max = popt[2]
    return popt, interp_c50, fitted_p_max
    
def bootstrap_fit(lin_ax,responses,n_boot=1000):
    
    

    params = np.zeros((n_boot,3))
    params[:] = np.nan
    interp_c50_array = []
    for i in range(n_boot):
        resampled = np.apply_along_axis(np.random.choice,0,responses,size=responses.shape[0],replace=True)
        means = np.nanmean(resampled,axis=0)
        popt,interp_c50,fitted_p_max = fit_naka_rushton(lin_ax,means,p_false=means[0])
        params[i,:] = popt
        interp_c50_array.append(interp_c50)
    return params, interp_c50_array

def bootstrap_means(lin_ax,responses,n_boot=1000):
    
    means = np.zeros((n_boot,3))
    means[:] = np.nan
    interp_c50_array = []
    for i in range(n_boot):
        resampled = np.apply_along_axis(np.random.choice,0,responses,size=responses.shape[0],replace=True)
        means[i,:]=np.nanmean(resampled,axis=0)
        
    return means

In [ ]:
hub = WheelDetectionExperimentHub()
# load from session list
# hub.set_data(all_sessions,
#              load_sessions=True,
#              make_summary=True)
# hub.data.write_parquet("250206_experiment_data.parquet")

# load from saved data
all_data = pl.read_parquet(DATA_PATH)
hub.set_data(all_data,
             load_sessions=True,
             make_summary=True)

In [ ]:
areas = ["V1"]
stim_combination = "0.04cpd_8.0Hz"
all_data = {}
for a in areas:
    df = hub.filter_by_areas(a,
                        strict_performance=True,
                        stim_combination=stim_combination,
                        isCNO=False)
    all_data[a] = df
hub.make_summary_data(df)

In [ ]:
aggregator = WheelDetectionGroupedAggregator()

area_matrices = {}
for k,v in all_data.items():
    aggregator.set_data(data=v)
    aggregator.group_data(
        group_by=[
            "stim_side",
            "animalid",
            "contrast",
            "opto_pattern",
            "stim_type"
        ]
    )
    aggregator.calculate_hit_rates()
    aggregator.calculate_opto_pvalues()
    
    qq = (aggregator.grouped_data
         .filter((pl.col("contrast") == 0) & (pl.col("opto_pattern") == -1))
         .group_by(["animalid"])
         .agg([
             pl.col("hit_count").sum(),
             pl.col("count").sum()
         ])
    ).sort("animalid")
    catch_trials = qq.with_columns(
        (pl.col("hit_count") / pl.col("count")).alias("baseline_hr")
    )
    
    _data = aggregator.grouped_data.join(
        catch_trials.select(["animalid","baseline_hr"]),
        how="inner",
        left_on= ["animalid"],
        right_on=["animalid"],
    )
    
    plot_data = _data.drop_nulls(["stim_side"]).filter(pl.col("stim_side") != "ipsi")

    
    q = (plot_data.group_by(["contrast","opto_pattern"])
            .agg(
                [pl.col("animalid"),
                 pl.col("hit_rate"),
                 pl.col("count"),
                 pl.col("baseline_hr"),
                 pl.col("median_hit_reaction_times")
                ]
            )).sort("contrast","opto_pattern")
    
    
    data_mat = nan_generator((
        plot_data.n_unique("animalid"),     # 
        plot_data.n_unique("contrast"),     # contrasts
        2,                                  # control, opto
        2                                   # [hit_rate, count]
    ))
    

    for opto_tup in make_subsets(q,["opto_pattern"],start_enumerate=0):
        opto_idx = opto_tup[0]
        opto_df = opto_tup[-1]
        for contrast_tup in make_subsets(opto_df,"contrast",start_enumerate=0):
            contrast = contrast_tup[1]
            contrast_idx = CONTRAST_IDX[contrast]
            contrast_df = contrast_tup[-1]
            
            hr = contrast_df["hit_rate"].explode().to_numpy()
            count = contrast_df["count"].explode().to_numpy()
            
            # b_hr = contrast_df["baseline_hr"].explode().to_numpy()
            
            # if contrast != 0:
            #     #percent_correct, floored at 0
            #     _pc = (hr - b_hr)/np.max(hr-b_hr)
            #     prcnt_corr = np.array([max(i,0) for i in _pc])
            # else:
            #     prcnt_corr = b_hr
            temp = np.hstack((hr.reshape(-1,1),
                                count.reshape(-1,1)))
            data_mat[:,contrast_idx,opto_idx,:] = temp
                                                                
    area_matrices[k] = data_mat
                

## Plotting

In [ ]:
do_fit="naka-rushton"
n_boot = 1000


cm = 1/2.54
set_style("print") 
f,ax = plt.subplots(1,
                     len(area_matrices),
                     figsize=(20*cm,12*cm))

stypes = ["0.04cpd_8.0Hz"]
lin_ax = [0,1,2]
log_ax = [0.1,12.5,50]
do_log=True

xax = log_ax if do_log else lin_ax

custom_handles = []

for area in area_matrices.keys():
    stype_slice = area_matrices[area]
    
    
    for m_idx in range(stype_slice.shape[2]): 
        manip_slice = stype_slice[:,:,m_idx,0] # use only hr and not count
        
        avg_hr = np.nanmean(manip_slice,axis=0)
        sem_hr = stats.sem(manip_slice,axis=0, nan_policy="omit")
        
        # params, interp_c50_array = bootstrap_fit(lin_ax,manip_slice,n_boot=n_boot)
        # means = bootstrap_means(lin_ax,manip_slice,n_boot=n_boot)
        boot_res = stats.bootstrap((manip_slice,),np.nanmean,vectorized=True,n_resamples=1000,paired=True,confidence_level=0.95)
        
        # _fit_data = np.vstack((np.array(lin_ax).reshape(1,-1), 
        #                        counts.reshape(1,-1), 
        #                        hr.reshape(1,-1)))
        # confs = np.percentile(means,[2.5,97.5],axis=0)
        

        popt = [None, None, None]
        interp_c50 = None
        is_fitted = True
        try:                
            if do_fit == "sigmoid":
                fit_func = sigmoid
                bounds = ([0.5, 0, 1], [1.5, 1, 5]) # [x0,L,k]
                fit_func_partial = partial(fit_func,p_false=avg_hr[0])
                popt,pcov = curve_fit(fit_func_partial, 
                                        xax, 
                                        hr, 
                                        p0=[1,0.5,3], 
                                        bounds=bounds, 
                                        maxfev=5000)
                
                interp_c50 = np.interp(popt[0],xax,[0,0.125,0.5])
                fitted_p_max = popt[1]

            elif do_fit == "naka-rushton":
                fit_func = naka_rushton
                bounds = ([1,1,0],[30,3,1]) # [c50,n,p_max]([0.5, 1, 0], [1.5, 5, 1])
                popt_avg, interp_c50_avg,fitted_p_max_avg = fit_naka_rushton(xax,avg_hr,p_false=avg_hr[0],bounds=bounds)

                
        except RuntimeError as e:
            is_fitted = False
        
        # jitter = np.random.random(len(lin_ax))*0.01
        # sc = ax.errorbar(lin_ax+jitter,
        #                  hr,
        #                  yerr=(hr+semm,hr-semm),
        #                  color="k",
        #                  linewidth=0,
        #                  elinewidth=1,
        #                  marker=MANIP_MARKER[col_idx],
        #                  label=f"{MANIP_LABEL[col_idx]} C50={round(interp_c50,2)}, pmax={round(fitted_p_max*100,2)}",
        #                  zorder=2,)
        
        # the actual avg dots
        sc = ax.scatter(xax,avg_hr,
                        color="k",
                        label=f"{MANIP_LABEL[m_idx]} C50={round(interp_c50_avg,2)}, pmax={round(fitted_p_max_avg*100,2)}",
                        marker=MANIP_MARKER[m_idx],
                        zorder=2)
        print("\n")
        print(area,MANIP_LABEL[m_idx])
        print("avg HR: ", avg_hr)
        print("95CI-:",boot_res.confidence_interval.low)
        print("95CI+:",boot_res.confidence_interval.high)
        print("SEM",sem_hr)
        ax.vlines(xax,ymin=boot_res.confidence_interval.low,ymax=boot_res.confidence_interval.high,
                    color="k")
        
        
        
        # fit to 95% CI's
        
        if area == "V1":
            ax.set_ylabel("Hit rate (%)")
        
        if is_fitted:
            x_fit1 = np.linspace(xax[0], xax[1], 50)
            x_fit2 = np.linspace(xax[1], xax[2], 50)
            x_fit = np.concatenate((x_fit1,x_fit2[1:]))
            # y_fit = weibull(params, x_fit)
            y_fit_avg = fit_func(x_fit, *popt_avg, p_false=avg_hr[0])
            ax.plot(x_fit,y_fit_avg,
                    linewidth=2,
                    color=STYPE_COLOR["0.04cpd_8.0Hz"],
                    zorder=0
                    )
            
            
            # y_fit_upper = fit_func(x_fit,*confs[1,:],p_false=0)
            # y_fit_lower = fit_func(x_fit,*confs[0,:],p_false=0)
            
            # ax.fill_between(x_fit,
            #                 y_fit_avg-y_fit_lower,
            #                 y_fit_avg+y_fit_upper,
            #                 alpha=0.2)
            
            ax.legend(loc="upper left",frameon=False,fontsize=10)
    
    if do_log:
        ax.set_xscale("log")
    ax.set_title(f"0.04cpd_8.0Hz_{area}")
    ax.set_xlabel("Contrast (%)")
    # ax.set_xticks(xax)
    # ax.set_xticklabels([0,12.5,50])
    ax.set_ylim([0,1])
    ax.set_xlim([-0.5,100])
    ax.set_yticks([0,0.25,0.5,0.75,1.0])
    ax.set_yticklabels([0,25,50,75,100])
        
        
print(custom_handles)
f.suptitle(f"{do_fit}")
f.legend(handles=custom_handles, ncol=len(MANIP_MARKER),loc='lower center')